## 1. Problem Statement

- Objective
Classify Flipkart product reviews into Positive or Negative sentiment and identify patterns contributing to customer satisfaction or dissatisfaction.

- ML Type: Supervised Learning
- Task: Text Classification
- Metric: F1-score (important for class imbalance)

## 2. Data Collection

🔸 Already done
- Data engineers scraped Flipkart reviews
- We are provided with a CSV file
- No scraping required

## 3. Data Loading / Data Ingestion
- Purpose:
    - Bring raw data into the ML pipeline
    - Validate schema
    - Loads the dataset into a DataFrame
    - DataFrame = table (rows & columns)
    - Displays first 5 rows to verify data is loaded correctly

In [18]:
import pandas as pd # Imports pandas library to work with tabular data

df = pd.read_csv("data.csv")
df.head()


,Reviewer Name,Review Title,Place of Review,Up Votes,Down Votes,Month,Review text,Ratings
0,Kamal Suresh,Nice product,"Certified Buyer, Chirakkal",889.0,64.0,Feb 2021,"Nice product, good quality, but price is now r...",4
1,Flipkart Customer,Don't waste your money,"Certified Buyer, Hyderabad",109.0,6.0,Feb 2021,They didn't supplied Yonex Mavis 350. Outside ...,1
2,A. S. Raja Srinivasan,Did not meet expectations,"Certified Buyer, Dharmapuri",42.0,3.0,Apr 2021,Worst product. Damaged shuttlecocks packed in ...,1
3,Suresh Narayanasamy,Fair,"Certified Buyer, Chennai",25.0,1.0,NaN,"Quite O. K. , but nowadays the quality of the...",3
4,ASHIK P A,Over priced,NaN,147.0,24.0,Apr 2016,Over pricedJust â?¹620 ..from retailer.I didn'...,1


## 4. Data Understanding (EDA)
- Number of rows & columns
- Missing values
- Data types

In [19]:
df.shape # returns number of rows and columns

(8518, 8)

In [20]:
df.columns # Helps us understand available features

Index(['Reviewer Name', 'Review Title', 'Place of Review', 'Up Votes',
       'Down Votes', 'Month', 'Review text', 'Ratings'],
      dtype='object')

In [21]:
df.isnull().sum() # Finds columns with missing data

Reviewer Name       10
Review Title        10
Place of Review     50
Up Votes            10
Down Votes          10
Month              465
Review text          8
Ratings              0
dtype: int64

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8518 entries, 0 to 8517
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Reviewer Name    8508 non-null   object 
 1   Review Title     8508 non-null   object 
 2   Place of Review  8468 non-null   object 
 3   Up Votes         8508 non-null   float64
 4   Down Votes       8508 non-null   float64
 5   Month            8053 non-null   object 
 6   Review text      8510 non-null   object 
 7   Ratings          8518 non-null   int64  
dtypes: float64(2), int64(1), object(5)
memory usage: 532.5+ KB


## Target understanding
- Usually:
    - Rating ≥ 4 → Positive
    - Rating ≤ 2 → Negative

In [23]:
df['Ratings'].value_counts()

Ratings
5    5080
4    1746
1     769
3     615
2     308
Name: count, dtype: int64

## 5. Data Cleaning
### Handle missing values

In [24]:
df = df.dropna(subset=['Review text', 'Ratings'])
df.isna().sum()

Reviewer Name        2
Review Title         2
Place of Review     42
Up Votes             2
Down Votes           2
Month              457
Review text          0
Ratings              0
dtype: int64

### Create sentiment label

In [25]:
def sentiment_label(rating):
    if rating >= 4:
        return 1
    elif rating <= 2:
        return 0
    else:
        return None

df['sentiment'] = df['Ratings'].apply(sentiment_label)
df = df.dropna(subset=['sentiment'])
df['sentiment']

0       1.0
1       0.0
2       0.0
4       0.0
5       1.0
       ... 
8504    1.0
8506    1.0
8507    1.0
8508    1.0
8509    0.0
Name: sentiment, Length: 7895, dtype: float64

## 6. Data Annotation (if required)

- Implicit annotation
    - Ratings are used as weak labels
    - No manual annotation needed
- This is common in real-world NLP projects.

## 7. Feature Engineering
### Text preprocessing

In [26]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return " ".join(words)

df['clean_review'] = df['Review text'].apply(clean_text)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kusum\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\kusum\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [27]:
stop_words = set(stopwords.words('english'))

# Remove important negation words from stopwords
negation_words = {"not", "no", "nor", "n't"}
stop_words = stop_words - negation_words

In [28]:
ngram_range=(1,2)

In [29]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)  # important
)

In [30]:
def clean_text(text):
    text = text.lower()
    
    # keep "not"
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    
    words = text.split()
    
    words = [
        lemmatizer.lemmatize(w)
        for w in words
        if w not in stop_words
    ]
    
    return " ".join(words)

In [31]:
df[df['clean_review'].str.contains("not good", na=False)].head()

,Reviewer Name,Review Title,Place of Review,Up Votes,Down Votes,Month,Review text,Ratings,sentiment,clean_review


In [32]:
def handle_negation(text):
    text = text.replace("not good", "not_good")
    text = text.replace("not bad", "not_bad")
    return text

## 8. Feature Selection / Dimensionality Reduction
### TF - IDF Vectorization
- TF-IDF already performs implicit feature selection by importance.

In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X = tfidf.fit_transform(df['clean_review'])
y = df['sentiment']


## 9. Data Splitting
- why
    - Prevent data leakage
    - Test generalization

In [34]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## 10. Model Selection
- Models considered:
    - Logistic Regression
    - Naive Bayes
    - Linear SVM
- Start simple and interpretable.

## 11. Model Training

In [35]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


## 12. Hyperparameter Tuning

In [36]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.1, 1, 10]
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid.fit(X_train, y_train)

,estimator,LogisticRegre...max_iter=1000)
,param_grid,"{'C': [0.1, 1, ...]}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


## 13. Model Evaluation
- Why here?
    - Predictions on unseen data
    - Measure real performance

In [37]:
from sklearn.metrics import classification_report

y_test_pred = grid.predict(X_test)
print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

         0.0       0.82      0.58      0.68       214
         1.0       0.94      0.98      0.96      1365

    accuracy                           0.93      1579
   macro avg       0.88      0.78      0.82      1579
weighted avg       0.92      0.93      0.92      1579



## 14. Error Analysis
- Check:
    - False positives
    - False negatives
    - Sarcasm, negation, short reviews
- Usually analyzed manually for NLP.
- y_test.values → NumPy array
- mask → NumPy boolean array
- Compatible with SciPy sparse matrix


- What this step means
    - Understanding why model fails.
    - In NLP, errors happen due to:
    - Sarcasm
    - Negation
    - Short reviews
- This guides future improvements.

In [38]:
import numpy as np

mask = (y_test.values != y_test_pred)
errors = X_test[mask]
errors

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 839 stored elements and shape (116, 5000)>

## 15. Model Validation
- What this step means

Ensuring model is:
- Stable
- Not overfitting

In THIS project

- Cross-validation inside GridSearchCV
- F1-score chosen correctly
- Validation ≠ evaluation.

- Cross-validation already done in GridSearchCV
- Best model selected via F1-score

In [39]:
best_model = grid.best_estimator_

## 16. Model Deployment
What this step means
- Making model usable by users.
- Model without deployment = useless.

- Deploy using :
    - Flask
    - Streamlit
    - AWS EC2

### Save model & vectorizer


In [40]:
import joblib

joblib.dump(best_model, "sentiment_model.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']

## 17. Monitoring
What this step means
- Watching model behavior after deployment.

Why needed
- Language changes
- Product changes
- User behavior changes

This is ML in production.
- Monitor prediction confidence
- Track drift in review language
- Monitor F1-score periodically

## 18. Retraining
What this step means
- Updating model with new data.

Why needed
- ML models decay over time.

Trigger retraining when:
- New product reviews arrive
- Model performance degrades
- Language patterns change

Pipeline loops back to:
- Data ingestion → training again.